Nama: Anis Al Munawwarah
Nim: 230212011
Praktikum Projec 4 || Data Science da Big Data

In [ ]:
#Section 1: Import Modul
#menyiapkan pustaka yang diperlukan untuk pemrosesan teks dan pembangunan model deep learning
#manipulasi data,
#--#
#--#
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix

# Perbaikan di bagian Keras/TensorFlow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences # Pengganti sequence.pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout, concatenate, Embedding
from tensorflow.keras.utils import plot_model

import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
#Section 2: Memuat dan Eksplorasi Dataset
#Membaca dataset
#----#
df = pd.read_csv('/content/all_agree.csv')

# Menampilkan 5 data teratas
print(df.head())

# Mengetahui jumlah sampel tiap class
print(df['label_score'].value_counts())

                                               title          label  \
0  Masuk Radar Pilwalkot Medan, Menantu Jokowi Be...  non-clickbait   
1  Malaysia Sudutkan RI: Isu Kabut Asap hingga In...  non-clickbait   
2  Viral! Driver Ojol di Bekasi Antar Pesanan Mak...      clickbait   
3  Kemensos Salurkan Rp 7,3 M bagi Korban Kerusuh...  non-clickbait   
4  MPR: Amandemen UUD 1945 Tak Akan Melebar ke Ma...  non-clickbait   

   label_score  
0            0  
1            0  
2            1  
3            0  
4            0  
label_score
0    5297
1    3316
Name: count, dtype: int64


In [ ]:
#Section 3: Feature Generation
#----#
# Fungsi untuk menghitung jumlah tanda baca tertentu
def find_punc(x, punc):
    count = 0
    for char in x:
        if char == punc:
            count += 1
    return count

# Fungsi untuk menghitung huruf besar, kecil, dan angka
def find_uppercase(x):
    count = 0
    for char in x:
        if char.isupper():
            count += 1
    return count

def find_lowercase(x):
    count = 0
    for char in x:
        if char.islower():
            count += 1
    return count

def find_number(x):
    return len(re.findall('[0-9]', x))

# Mengaplikasikan fungsi pada kolom 'title'
df['comma_count'] = df['title'].apply(lambda x: find_punc(x, ','))
df['fullstop_count'] = df['title'].apply(lambda x: find_punc(x, '.'))
# ... (lanjutkan untuk tanda baca lainnya seperti '?', '!', dll)
df['uppercase_count'] = df['title'].apply(find_uppercase)
df['lowercase_count'] = df['title'].apply(find_lowercase)
df['number_count'] = df['title'].apply(find_number)

# Cek dimensi data baru
print(df.shape)

(8613, 8)


In [ ]:
#Section 4: Membersihkan Data Teks
#----#
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
def clean(text):
    # 1. Mengubah ke huruf kecil
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    word_tokens = word_tokenize(text)
    text = ' '.join(word_tokens)
    return text
# Menerapkan fungsi
df['cleaned_title'] = df['title'].apply(clean)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
#Section 5: Train-Test Split
#Membagi data menjadi data latih (80%) dan data uji (20%)

# Mendefinisikan fitur (X) dan label (y)
X = df.iloc[:, 3:] # Kolom fitur mulai dari comma_count ke kanan
y = df.iloc[:, 2]  # Kolom label_score

# Melakukan splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=77)

# Menampilkan hasil splitting
print("--- Hasil Splitting Data ---")
print(f"Total data asli: {len(df)}")
print("-" * 17)
print(f"X_train (Fitur Training): {X_train.shape}")
print(f"y_train (Target Training): {y_train.shape}")
print("-" * 17)
print(f"X_test  (Fitur Testing) : {X_test.shape}")
print(f"y_test  (Target Testing) : {y_test.shape}")

--- Hasil Splitting Data ---
Total data asli: 8613
-----------------
X_train (Fitur Training): (6890, 6)
y_train (Target Training): (6890,)
-----------------
X_test  (Fitur Testing) : (1723, 6)
y_test  (Target Testing) : (1723,)


In [ ]:
#Section 6: Preprocessing Lanjutan untuk Teks
#Proses integer-encoding dan zero-padding agar semua input teks memiliki panjang yang sama (19 kata).

# Memisahkan fitur teks dan fitur tanda baca
X_train_title = X_train['cleaned_title']
X_test_title = X_test['cleaned_title']
X_train_punc = X_train.drop('cleaned_title', axis=1)
X_test_punc = X_test.drop('cleaned_title', axis=1)

# Inisialisasi Tokenizer dan fitting pada data train
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train_title)

# Konversi teks menjadi urutan angka
X_train_title = tokenizer.texts_to_sequences(X_train_title)
X_test_title = tokenizer.texts_to_sequences(X_test_title)

# Menyimpan ukuran vocabulary
vocab_size = len(tokenizer.word_index) + 1

# Menyamakan panjang kalimat (Padding) ke 19 kata
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Terapkan langsung tanpa 'sequence.'
X_train_title = pad_sequences(X_train_title, maxlen=19)
X_test_title = pad_sequences(X_test_title, maxlen=19)

print("--- Hasil Preprocessing Teks ---")
print(f"Ukuran Vocabulary: {vocab_size}")
print(f"Bentuk Array Train: {X_train_title.shape}")

# Menampilkan contoh kalimat pertama setelah di-padding
print("\nContoh Sequence pada data train (indeks 0):")
print(X_train_title[0])

--- Hasil Preprocessing Teks ---
Ukuran Vocabulary: 12082
Bentuk Array Train: (6890, 19)

Contoh Sequence pada data train (indeks 0):
[   0    0    0    0    0    0    0    0    0   56 6278 2264   84  143
 2700  258   53  144  555]


In [ ]:
#Section 7: Pembobotan Class
#Mengatasi ketidakseimbangan data (imbalanced class) dengan memberikan bobot lebih besar pada class minoritas

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Konversi ke dictionary agar bisa dibaca Keras
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)

{0: np.float64(0.8086854460093896), 1: np.float64(1.3098859315589353)}


In [ ]:
#Section 8: Membangun Arsitektur LSTM (Awal)
#Inisialisasi layer input dan embedding untuk cabang LSTM

tf.random.set_seed(99)

# Membangun input LSTM
lstm_input = Input(shape=(19,))
emb = Embedding(input_dim=vocab_size + 1, output_dim=32, input_length=19)(lstm_input)
# ... (lanjutkan ke layer LSTM dan Dense sesuai arsitektur di file)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
